# BigQuery Performance Benchmark

This notebook benchmarks two key BigQuery patterns: **External Tables (BigLake)** and **Native Managed Tables**.

We will explore the performance differences between querying data residing in Google Cloud Storage (via BigLake) versus loading that same data into BigQuery's native managed storage.

## Scenarios

1.  **BigLake External Tables**: Querying Parquet data directly in GCS. Zero-copy, instant access.
2.  **Native BigQuery Tables**: Loading Parquet data into managed BigQuery storage. Optimized for performance and high concurrency.

## Prerequisites

This notebook depends on the ~550GB Parquet dataset generated by the **Dataproc Serverless Performance Benchmark**.
Ensure you have completed the data generation step in the main Dataproc demo.

In [ ]:
# Set your environment variables here
GOOGLE_PROJECT_ID = "your-gcp-project-id"
REGION = "your-region" # e.g., us-central1
GCS_BUCKET = "your-unique-bucket-name"

# Constants
BIGLAKE_CONNECTION_ID = "DEFAULT" # Use Default or Replace with your connection ID

In [ ]:
from google.cloud import bigquery
import time

client = bigquery.Client(project=GOOGLE_PROJECT_ID, location=REGION)

print(f"Initialized BigQuery client for project: {GOOGLE_PROJECT_ID} in {REGION}")

## 1. Create BigLake External Tables

First, we create a dataset `sales_data_biglake` and external tables pointing to the Parquet data in GCS.
This allows us to query the data immediately without moving it.

In [ ]:
# Create Dataset
dataset_id_biglake = f"{GOOGLE_PROJECT_ID}.sales_data_biglake"
dataset_biglake = bigquery.Dataset(dataset_id_biglake)
dataset_biglake.location = REGION
dataset_biglake.description = "Dataset for BigLake external tables querying GCS Parquet data"

try:
    client.create_dataset(dataset_biglake, exists_ok=True)
    print(f"Dataset {dataset_id_biglake} created or already exists.")
except Exception as e:
    print(f"Error creating dataset: {e}")

In [ ]:
# Create External Tables
sql_setup_biglake = f"""
CREATE OR REPLACE EXTERNAL TABLE `{GOOGLE_PROJECT_ID}.sales_data_biglake.customer_master`
(
  customer_id INT64,
  name FLOAT64,
  state STRING,
  signup_date TIMESTAMP
)
WITH CONNECTION {BIGLAKE_CONNECTION_ID}
OPTIONS (
  format = 'PARQUET',
  uris = ['gs://{GCS_BUCKET}/dataproc-benchmark-data/customer_master.parquet/*'],
  metadata_cache_mode = 'AUTOMATIC',
  max_staleness = INTERVAL 30 MINUTE,
  enable_list_inference = TRUE
);

CREATE OR REPLACE EXTERNAL TABLE `{GOOGLE_PROJECT_ID}.sales_data_biglake.sales_transactions`
(
  transaction_id INT64,
  customer_id INT64,
  product_id INT64,
  transaction_amount FLOAT64,
  `timestamp` TIMESTAMP
)
WITH PARTITION COLUMNS (
  region STRING
)
WITH CONNECTION {BIGLAKE_CONNECTION_ID}
OPTIONS (
  format = 'PARQUET',
  uris = ['gs://{GCS_BUCKET}/dataproc-benchmark-data/sales_transactions.parquet/*'],
  hive_partition_uri_prefix = 'gs://{GCS_BUCKET}/dataproc-benchmark-data/sales_transactions.parquet',
  metadata_cache_mode = 'AUTOMATIC',
  max_staleness = INTERVAL 30 MINUTE,
  enable_list_inference = TRUE
);
"""

job = client.query(sql_setup_biglake)
job.result()  # Wait for the job to complete
print("BigLake external tables created successfully.")

## 2. Create Native BigQuery Tables

Now, we create a standard native dataset `sales_data_native` and load the same data into it.
Native tables use BigQuery's proprietary storage format (Capacitor), which generally offers better performance, caching, and features like Time Travel.

In [ ]:
# Create Dataset
dataset_id_native = f"{GOOGLE_PROJECT_ID}.sales_data_native"
dataset_native = bigquery.Dataset(dataset_id_native)
dataset_native.location = REGION
dataset_native.description = "Dataset with native managed storage"
client.create_dataset(dataset_native, exists_ok=True)

print(f"Dataset created: {dataset_id_native}")

In [ ]:
# Load Data (This may take a few minutes for 550GB)
sql_load_data = f"""
LOAD DATA OVERWRITE `{GOOGLE_PROJECT_ID}.sales_data_native.customer_master`
(
  customer_id INT64,
  name STRING,
  state STRING,
  signup_date TIMESTAMP
)
CLUSTER BY customer_id
FROM FILES (
  format = 'PARQUET',
  uris = ['gs://{GCS_BUCKET}/dataproc-benchmark-data/customer_master.parquet/*']
);

LOAD DATA OVERWRITE `{GOOGLE_PROJECT_ID}.sales_data_native.sales_transactions`
(
  transaction_id INT64,
  customer_id INT64,
  product_id INT64,
  transaction_amount FLOAT64,
  `timestamp` TIMESTAMP
)
CLUSTER BY customer_id
FROM FILES (
  format = 'PARQUET',
  uris = ['gs://{GCS_BUCKET}/dataproc-benchmark-data/sales_transactions.parquet/*'],
  hive_partition_uri_prefix = 'gs://{GCS_BUCKET}/dataproc-benchmark-data/sales_transactions.parquet'
)
WITH PARTITION COLUMNS (
  region STRING
);
"""

print("Starting data load jobs... this involves ingesting ~550GB and might take a minute.")
job = client.query(sql_load_data)
job.result()
print("Data loaded successfully into native dataset.")

## 3. Run Analysis Queries & Benchmark

We will now run the exact same analytic query against both the External Table (GCS) and the Native Table (Managed Columns).
Observe the difference in **Execution Time** and **Slot Usage** (if available).

In [ ]:
def run_and_bench(query, description):
    print(f"--- Running: {description} ---")
    # Disable cache to ensure we measure actual query processing time every run
    job_config = bigquery.QueryJobConfig(use_query_cache=False)
    
    start_time = time.time()
    job = client.query(query, job_config=job_config)
    result = job.result()
    end_time = time.time()
    
    elapsed = end_time - start_time
    bytes_processed_gb = job.total_bytes_processed / (1024 ** 3)
    
    print(f"Elapsed Time:    {elapsed:.2f} seconds")
    print(f"Bytes Processed: {bytes_processed_gb:.2f} GB")
    print(f"Job ID:          {job.job_id}")

# Query Template
query_template = """
SELECT
  s.region,
  c.state,
  CASE
    WHEN c.state IN ('CA', 'NY') THEN 'Gold'
    ELSE 'Silver'
  END AS customer_tier,
  SUM(s.transaction_amount) AS total_sales,
  AVG(s.transaction_amount) AS average_sale_amount,
  COUNT(DISTINCT s.customer_id) AS distinct_customers
FROM
  `{DATASET}.sales_transactions` AS s
JOIN
  `{DATASET}.customer_master` AS c
ON
  s.customer_id = c.customer_id
GROUP BY
  region,
  state,
  customer_tier
ORDER BY
  total_sales DESC;
"""

In [ ]:
# 1. BigLake External Table Benchmark
query_biglake = query_template.format(DATASET=f"{GOOGLE_PROJECT_ID}.sales_data_biglake")
run_and_bench(query_biglake, "BigLake External Table (GCS)")

In [ ]:
# 2. Native Managed Table Benchmark
query_native = query_template.format(DATASET=f"{GOOGLE_PROJECT_ID}.sales_data_native")
run_and_bench(query_native, "Native Managed Table")

## Conclusion & Takeaways

1.  **BigLake** provides incredible flexibility. You queried 500GB+ of data directly in GCS without any prior ingestion. This is perfect for ad-hoc exploration or when freshness is paramount.
2.  **Native Tables** should generally be faster for repeated analytic workloads. BigQuery's storage format (Capacitor) allows for aggressive filtering and aggregation optimizations not possible with raw Parquet.

In [ ]:
# Optional Cleanup
# client.delete_dataset(dataset_id_biglake, delete_contents=True, not_found_ok=True)
# client.delete_dataset(dataset_id_native, delete_contents=True, not_found_ok=True)
# print("Datasets deleted.")